# Embedding Techniques Comparison
Compare six embedding methods for candidate ranking by semantic similarity to target keywords.

In [ ]:
import sys
from pathlib import Path

try:
    ROOT = Path(__file__).resolve().parent.parent  # src/ -> project root (running as script)
except NameError:
    ROOT = Path().resolve().parent                 # notebooks/ -> project root (running as notebook)

sys.path.insert(0, str(ROOT))

from IPython.display import Image, display
from src.config import DATA_FILE
from src.data_loader import load_data
from src.preprocessing import preprocess
from src.compare_embeddings import METHODS, visualize

df = load_data(ROOT / DATA_FILE)
df = preprocess(df)
df.head()

,id,job_title,location,connection,fit,job_title_clean,connections_raw,connections_norm
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN,2019 C.T. Bauer College of Business Graduate (...,85,0.170
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN,Native English Teacher at EPIK (English Progra...,500,1.000
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN,Aspiring Human Resources Professional,44,0.088
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN,People Development Coordinator at Ryan,500,1.000
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN,Advisory Board Member at Celal Bayar University,500,1.000


## Inspect raw data

In [ ]:
print(df.shape)
print(df.dtypes)
df["connection"].value_counts()

(104, 8)
id                    int64
job_title            object
location             object
connection           object
fit                 float64
job_title_clean      object
connections_raw       int64
connections_norm    float64
dtype: object


connection
500+     44
85        7
61        7
44        6
1         5
2         4
4         2
7         2
57        2
390       2
103       1
48        1
18        1
71        1
19        1
415       1
9         1
64        1
39        1
155       1
349       1
174       1
40        1
50        1
268       1
455       1
52        1
409       1
212       1
16        1
5         1
82        1
49        1
Name: count, dtype: int64

## Preprocess

Three steps:
1. `job_title_clean` — title kept as-is (transformer models handle raw text better than stripped tokens)
2. `connections_raw` — parse "500+" → 500, invalid → 0
3. `connections_norm` — divide by 500, clip to [0, 1] so it can be blended with cosine similarity scores

In [ ]:
df[["job_title", "job_title_clean", "connections_raw", "connections_norm"]].head(10)

,job_title,job_title_clean,connections_raw,connections_norm
0,2019 C.T. Bauer College of Business Graduate (...,2019 C.T. Bauer College of Business Graduate (...,85,0.170
1,Native English Teacher at EPIK (English Progra...,Native English Teacher at EPIK (English Progra...,500,1.000
2,Aspiring Human Resources Professional,Aspiring Human Resources Professional,44,0.088
3,People Development Coordinator at Ryan,People Development Coordinator at Ryan,500,1.000
4,Advisory Board Member at Celal Bayar University,Advisory Board Member at Celal Bayar University,500,1.000
5,Aspiring Human Resources Specialist,Aspiring Human Resources Specialist,1,0.002
6,Student at Humber College and Aspiring Human R...,Student at Humber College and Aspiring Human R...,61,0.122
7,HR Senior Specialist,HR Senior Specialist,500,1.000
8,Student at Humber College and Aspiring Human R...,Student at Humber College and Aspiring Human R...,61,0.122
9,Seeking Human Resources HRIS and Generalist Po...,Seeking Human Resources HRIS and Generalist Po...,500,1.000


## Embedding Methods Overview

Each method embeds candidate job titles and target keywords into vector space,
then computes cosine similarity. The candidate's score = max similarity across all targets.

| Method    | Type              | Trained on          | Key property |
|-----------|-------------------|---------------------|--------------|
| TF-IDF    | Bag-of-words      | This corpus          | Exact word match, weighted by rarity |
| Word2Vec  | Word embeddings   | This corpus          | Context-based, avg pooling |
| FastText  | Subword embeddings| This corpus          | Handles rare/misspelled words |
| GloVe     | Word embeddings   | Wikipedia + Gigaword| Pretrained, reliable vectors |
| BERT      | Transformer       | Large pretraining   | Full sentence context, symmetric |
| E5-small  | Transformer       | Large pretraining   | Designed for retrieval, asymmetric |

## TF-IDF

**What it is:** Bag-of-words method. Gives each word a weight based on how often it appears
in a document (TF) and how rare it is across all documents (IDF). No understanding of meaning.

**Advantage:** Simple, fast, interpretable.

**Disadvantages:**
- Long titles get penalized: "Aspiring HR Manager | Graduating May 2020 | Seeking Entry-Level Position"
  has its signal diluted by extra words, so cosine similarity with the short target is lower
  even though the candidate is clearly relevant.
- Vocabulary mismatch: "HR" and "Human Resources" are treated as completely unrelated.
  "seek" and "seeking" are different tokens even after lemmatization.

In [ ]:
df["fit_tfidf"] = METHODS["TF-IDF"](df)
df[["id", "job_title", "fit_tfidf"]].sort_values("fit_tfidf", ascending=False).head(10)

,id,job_title,fit_tfidf
32,33,Aspiring Human Resources Professional,0.749144
96,97,Aspiring Human Resources Professional,0.749144
20,21,Aspiring Human Resources Professional,0.749144
45,46,Aspiring Human Resources Professional,0.749144
57,58,Aspiring Human Resources Professional,0.749144
16,17,Aspiring Human Resources Professional,0.749144
2,3,Aspiring Human Resources Professional,0.749144
5,6,Aspiring Human Resources Specialist,0.691187
35,36,Aspiring Human Resources Specialist,0.691187
48,49,Aspiring Human Resources Specialist,0.691187


## Word2Vec

**What it is:** Trains a shallow neural network to predict surrounding words (skip-gram)
or the current word from context (CBOW). Words appearing in similar contexts get similar vectors.
A candidate's title vector = average of its word vectors.

**Advantage over TF-IDF:** Words don't need to match exactly.
"HR" and "human resources" may end up nearby if they appear in similar contexts.

**Disadvantages:**
- **Negation failure:** Averaging ignores negation.
  "not interested in management" and "interested in management" produce nearly the same vector
  because "not" is a tiny, direction-agnostic word that barely shifts the mean.
- **Small corpus:** With only ~100 job titles, Word2Vec can't learn reliable relationships.
  "aspiring" and "seeking" will have near-random vectors relative to each other.
- **Averaging loses word order:** "Human Resources Manager" and "Manager Human Resources"
  produce identical vectors.

In [ ]:
df["fit_w2v"] = METHODS["Word2Vec"](df)
df[["id", "job_title", "fit_w2v"]].sort_values("fit_w2v", ascending=False).head(10)

,id,job_title,fit_w2v
69,70,"Retired Army National Guard Recruiter, office ...",0.998939
91,92,Seeking employment opportunities within Custom...,0.998447
102,103,Always set them up for Success,0.998405
79,80,Junior MES Engineer| Information Systems,0.998097
84,85,RRP Brand Portfolio Executive at JTI (Japan To...,0.998055
74,75,"Nortia Staffing is seeking Human Resources, Pa...",0.997988
92,93,Admissions Representative at Community medical...,0.997960
89,90,Undergraduate Research Assistant at Styczynski...,0.997701
82,83,HR Manager at Endemol Shine North America,0.997577
85,86,Information Systems Specialist and Programmer ...,0.997571


## FastText

**What it is:** Extends Word2Vec with character-level n-gram embeddings.
"resources" → "res", "reso", "resou", ... The word vector = average of all its n-gram vectors.

**Advantage over Word2Vec:** Handles rare and unseen words.
"HR-Specialist" or "Resourcing" can still get a meaningful vector by composing
subword pieces shared with known words like "resource" and "resources".

**Disadvantages:**
- Still trained on a small corpus (~100 titles) — n-gram patterns are poorly calibrated.
- Averaging still loses word order, same as Word2Vec.
- **Cannot distinguish intent from surface similarity:**
  "looking for HR manager role" (candidate) and "hiring HR manager role" (employer)
  share the same words → nearly identical vectors. FastText has no way to tell them apart.
  BERT-style models handle this better by encoding full sentence context.

In [ ]:
df["fit_ft"] = METHODS["FastText"](df)
df[["id", "job_title", "fit_ft"]].sort_values("fit_ft", ascending=False).head(10)

,id,job_title,fit_ft
32,33,Aspiring Human Resources Professional,0.999964
45,46,Aspiring Human Resources Professional,0.999964
96,97,Aspiring Human Resources Professional,0.999964
20,21,Aspiring Human Resources Professional,0.999964
16,17,Aspiring Human Resources Professional,0.999964
57,58,Aspiring Human Resources Professional,0.999964
2,3,Aspiring Human Resources Professional,0.999964
28,29,Aspiring Human Resources Management student se...,0.999961
26,27,Aspiring Human Resources Management student se...,0.999961
29,30,Seeking Human Resources Opportunities,0.999954


## GloVe

**What it is:** Pretrained word embeddings trained on 6 billion tokens from Wikipedia
and Gigaword by factorizing a global word co-occurrence matrix.

**Advantage over Word2Vec/FastText on our corpus:** Trained on massive, diverse data —
"aspiring", "seeking", "human", "resources" all have well-calibrated vectors.
"HR" and "human resources" are likely close because they co-occur in the same Wikipedia contexts.

**Disadvantages:**
- **Domain gap:** Not trained on job postings or LinkedIn data.
  Jargon like "HRBP" or "talent acquisition" may be absent (OOV → zero vector).
- **Still word-level averaging:** Same word-order and negation blindness as Word2Vec.

In [ ]:
df["fit_glove"] = METHODS["GloVe"](df)
df[["id", "job_title", "fit_glove"]].sort_values("fit_glove", ascending=False).head(10)

,id,job_title,fit_glove
75,76,Aspiring Human Resources Professional | Passio...,0.798086
91,92,Seeking employment opportunities within Custom...,0.763749
85,86,Information Systems Specialist and Programmer ...,0.737385
83,84,Human Resources professional for the world lea...,0.734744
69,70,"Retired Army National Guard Recruiter, office ...",0.734384
65,66,Experienced Retail Manager and aspiring Human ...,0.722934
93,94,Seeking Human Resources Opportunities. Open t...,0.721806
102,103,Always set them up for Success,0.716034
13,14,2019 C.T. Bauer College of Business Graduate (...,0.709663
18,19,2019 C.T. Bauer College of Business Graduate (...,0.709663


## BERT (all-MiniLM-L6-v2)

**What it is:** Transformer model that encodes the entire sentence at once using self-attention.
Every token attends to every other token, so context is captured — "resources" in
"human resources" is represented differently than "resources" in "natural resources".

**Advantages:**
- Semantic understanding: matches "aspiring human resources" with "HR enthusiast looking for a role"
  even with no word overlap.
- Handles long titles: the full title is encoded as one vector — extra words don't dilute the signal.
- No preprocessing needed: handles punctuation, casing, and stopwords internally.

**Disadvantage — symmetric model:**
all-MiniLM-L6-v2 encodes both the candidate title and the target keyword the same way.
It does not distinguish between a "document to be retrieved" and a "search query".
For asymmetric retrieval (short query vs. longer document), E5 is better suited.

In [ ]:
df["fit_bert"] = METHODS["BERT"](df)
df[["id", "job_title", "fit_bert"]].sort_values("fit_bert", ascending=False).head(10)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,id,job_title,fit_bert
16,17,Aspiring Human Resources Professional,0.949807
2,3,Aspiring Human Resources Professional,0.949807
32,33,Aspiring Human Resources Professional,0.949807
20,21,Aspiring Human Resources Professional,0.949807
96,97,Aspiring Human Resources Professional,0.949807
45,46,Aspiring Human Resources Professional,0.949807
57,58,Aspiring Human Resources Professional,0.949807
59,60,Aspiring Human Resources Specialist,0.928035
48,49,Aspiring Human Resources Specialist,0.928035
5,6,Aspiring Human Resources Specialist,0.928035


## E5-small (intfloat/e5-small-v2)

**What it is:** Transformer model designed specifically for text retrieval.
Trained to distinguish between **queries** (what you search for) and **passages** (documents being searched).
The "passage: " and "query: " prefixes are mandatory — they are part of the model's training protocol.
Omitting them produces generic embeddings and significantly degrades similarity scores.

**How it differs from BERT:**
- BERT is **symmetric**: both sides encoded the same way.
- E5 is **asymmetric**: query and passage have separate learned representations,
  which better matches real retrieval — a short keyword and a long job title are different in nature.

**Advantages:**
- Purpose-built for retrieval: "query: aspiring human resources" pulls close to
  "passage: Aspiring HR Professional | Entry-Level Candidate" even without exact word overlap.
- Strong out-of-the-box performance — no fine-tuning needed on our data.

**Disadvantage:** e5-small-v2 is a compressed model (12M parameters).
e5-large-v2 would be more accurate but slower — for ~100 candidates the small variant is sufficient.

In [ ]:
df["fit_e5"] = METHODS["E5-small"](df)
df[["id", "job_title", "fit_e5"]].sort_values("fit_e5", ascending=False).head(10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,id,job_title,fit_e5
27,28,Seeking Human Resources Opportunities,0.945055
29,30,Seeking Human Resources Opportunities,0.945055
98,99,Seeking Human Resources Position,0.943834
45,46,Aspiring Human Resources Professional,0.930533
20,21,Aspiring Human Resources Professional,0.930533
32,33,Aspiring Human Resources Professional,0.930533
57,58,Aspiring Human Resources Professional,0.930533
16,17,Aspiring Human Resources Professional,0.930533
96,97,Aspiring Human Resources Professional,0.930533
2,3,Aspiring Human Resources Professional,0.930533


## Score Comparison Across All Methods

In [ ]:
score_cols = ["id", "job_title", "fit_tfidf", "fit_w2v", "fit_ft", "fit_glove", "fit_bert", "fit_e5"]
df[score_cols].sort_values("fit_e5", ascending=False).head(20)

,id,job_title,fit_tfidf,fit_w2v,fit_ft,fit_glove,fit_bert,fit_e5
27,28,Seeking Human Resources Opportunities,0.658595,0.993903,0.999954,0.000000,0.899172,0.945055
29,30,Seeking Human Resources Opportunities,0.658595,0.993903,0.999954,0.000000,0.899172,0.945055
98,99,Seeking Human Resources Position,0.638511,0.993051,0.999948,0.000000,0.904125,0.943834
45,46,Aspiring Human Resources Professional,0.749144,0.993827,0.999964,0.000000,0.949807,0.930533
20,21,Aspiring Human Resources Professional,0.749144,0.993827,0.999964,0.000000,0.949807,0.930533
32,33,Aspiring Human Resources Professional,0.749144,0.993827,0.999964,0.000000,0.949807,0.930533
57,58,Aspiring Human Resources Professional,0.749144,0.993827,0.999964,0.000000,0.949807,0.930533
16,17,Aspiring Human Resources Professional,0.749144,0.993827,0.999964,0.000000,0.949807,0.930533
96,97,Aspiring Human Resources Professional,0.749144,0.993827,0.999964,0.000000,0.949807,0.930533
2,3,Aspiring Human Resources Professional,0.749144,0.993827,0.999964,0.000000,0.949807,0.930533


## Visualize Embedding Space

Each method's embeddings are reduced to 2D using PCA and t-SNE.
- **Green dots** = candidates starred by the recruiter (relevant)
- **Light blue dots** = non-relevant candidates
- **Red stars** = target keywords (T1, T2)

A good embedding method should cluster green dots close to the red stars.

In [ ]:
visualize(df)

Computing embeddings for visualization...
  TF-IDF... done
  Word2Vec... done
  FastText... done
  GloVe... done
  BERT... 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


done
  E5-small... 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


done

Reducing with PCA...
Saved: C:\Users\D01173\Downloads\workspace\wzhTtO9tUf6Am2JF\src\embedding_space_pca.png
Reducing with t-SNE (slower)...


TypeError: TSNE.__init__() got an unexpected keyword argument 'max_iter'

### PCA — Linear projection (fast, shows global structure)

In [ ]:
display(Image(filename=str(ROOT / "embedding_space_pca.png")))

### t-SNE — Non-linear projection (slower, better cluster separation)

Note: distances *between* clusters in t-SNE are not meaningful — only local neighbourhood structure is.

In [ ]:
display(Image(filename=str(ROOT / "embedding_space_tsne.png")))